## Grouped Aggregations using Agg on a Spark Data Frame

In [2]:
%run './01 Validate Data Sets for Aggregations using Spark.ipynb'

24/03/24 19:51:47 WARN Utils: Your hostname, Qasim resolves to a loopback address: 127.0.1.1; using 172.30.54.131 instead (on interface eth0)
24/03/24 19:51:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/03/24 19:51:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/03/24 19:51:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/03/24 19:51:55 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
24/03/24 19:51:55 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


root
 |-- order_id: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- order_customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)

root
 |-- order_item_id: integer (nullable = true)
 |-- order_item_order_id: integer (nullable = true)
 |-- order_item_product_id: integer (nullable = true)
 |-- order_item_quantity: short (nullable = true)
 |-- order_item_subtotal: float (nullable = true)
 |-- order_item_product_price: float (nullable = true)



In [3]:
order_items_grouped = order_items \
    .select('order_item_order_id', 'order_item_quantity', 'order_item_subtotal') \
    .groupBy('order_item_order_id')
    #.show() does not work since grouoped data has no attribute show

In [5]:
from pyspark.sql.functions import sum, min, max, count

In [6]:
order_items_grouped. \
    agg(sum('order_item_quantity')). \
    show()

+-------------------+------------------------+
|order_item_order_id|sum(order_item_quantity)|
+-------------------+------------------------+
|                148|                       8|
|                463|                      13|
|                471|                       2|
|                496|                       7|
|               1088|                       3|
|               1580|                       5|
|               1591|                      10|
|               1645|                      14|
|               2366|                       3|
|               2659|                       8|
|               2866|                       5|
|               3175|                       4|
|               3749|                       3|
|               3794|                       5|
|               3918|                       9|
|               3997|                       4|
|               4101|                       1|
|               4519|                       2|
|            

In [7]:
order_items_grouped. \
    agg({'order_item_quantity': 'sum', 'order_item_subtotal': 'sum'}). \
    show()

+-------------------+------------------------+------------------------+
|order_item_order_id|sum(order_item_subtotal)|sum(order_item_quantity)|
+-------------------+------------------------+------------------------+
|                148|      479.99000549316406|                       8|
|                463|       829.9200096130371|                      13|
|                471|      169.98000717163086|                       2|
|                496|        441.950008392334|                       7|
|               1088|      249.97000885009766|                       3|
|               1580|      299.95001220703125|                       5|
|               1591|       439.8599967956543|                      10|
|               1645|       1509.790023803711|                      14|
|               2366|       299.9700012207031|                       3|
|               2659|       724.9100151062012|                       8|
|               2866|        569.960018157959|                  

In [8]:
from pyspark.sql.functions import round

In [16]:
    #withColumnRenamed('order_item_subtotal', round('order_item_quantity', 2)) does not work
    #Check help(withColumnRenamed)

order_items_grouped. \
    agg({'order_item_quantity': 'sum', 'order_item_subtotal': 'sum'}). \
    toDF('order_item_order_id', 'order_item_subtotal', 'order_item_quantity'). \
    withColumn('order_item_subtotal', round('order_item_subtotal', 2)). \
    show()

+-------------------+-------------------+-------------------+
|order_item_order_id|order_item_subtotal|order_item_quantity|
+-------------------+-------------------+-------------------+
|                148|             479.99|                  8|
|                463|             829.92|                 13|
|                471|             169.98|                  2|
|                496|             441.95|                  7|
|               1088|             249.97|                  3|
|               1580|             299.95|                  5|
|               1591|             439.86|                 10|
|               1645|            1509.79|                 14|
|               2366|             299.97|                  3|
|               2659|             724.91|                  8|
|               2866|             569.96|                  5|
|               3175|             209.97|                  4|
|               3749|             143.97|                  3|
|       

In [21]:
order_items_grouped. \
    agg(
        count('order_item_order_id').alias('total_orders'),
        round(sum('order_item_subtotal'), 2).alias('total_revenue'),
        min('order_item_quantity').alias('min_order'),
        max('order_item_quantity').alias('max_order')
    ). \
    show()

+-------------------+------------+-------------+---------+---------+
|order_item_order_id|total_orders|total_revenue|min_order|max_order|
+-------------------+------------+-------------+---------+---------+
|                148|           3|       479.99|        1|        5|
|                463|           4|       829.92|        1|        5|
|                471|           2|       169.98|        1|        1|
|                496|           5|       441.95|        1|        3|
|               1088|           2|       249.97|        1|        2|
|               1580|           1|       299.95|        5|        5|
|               1591|           3|       439.86|        1|        5|
|               1645|           5|      1509.79|        1|        5|
|               2366|           1|       299.97|        3|        3|
|               2659|           5|       724.91|        1|        3|
|               2866|           4|       569.96|        1|        2|
|               3175|           2|